# Proyek Klasifikasi Gambar: Open Images IT Asset Classification

- Nama: Agung Trisutaji Aprian
- Email: agung.trisutaji.aprian@gmail.com
- ID Dicoding: agungtrisutaji

## 1. Project Overview

Notebook ini adalah pipeline submission Dicoding **Belajar Fundamental Deep Learning** untuk klasifikasi gambar aset IT berbasis **Open Images V7 IT Asset Subset**. Revisi ini dibuat agar bisa dijalankan di **Google Colab dengan GPU**: data digabung dari satu sumber (`dataset/raw`), dibagi manual menjadi train/validation/test, model final `tf.keras.Sequential` dengan `Conv2D` dan pooling dilatih langsung, evaluasi test dihitung langsung, lalu model yang sama diexport ke SavedModel, TFLite, dan TFJS.

Sebelum menjalankan di Colab, pilih `Runtime > Change runtime type > T4 GPU` atau GPU lain. Jika repo/dataset belum ada di Colab, upload atau mount folder project yang berisi `configs/`, `dataset/raw/`, dan `dataset/metadata/openimages_crop_metadata.csv`.

## Import Semua Packages/Library yang Digunakan

Cell pertama mendeteksi apakah notebook berjalan di Google Colab, menampilkan GPU, dan menghentikan eksekusi lebih awal jika Colab belum memakai GPU.

In [1]:
from __future__ import annotations

import hashlib
import importlib.machinery
import json
import os
import random
import shutil
import subprocess
import sys
import types
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab")
    subprocess.run(["nvidia-smi"], check=True)
else:
    print("Running outside Google Colab")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

GPU_DEVICES = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU devices:", GPU_DEVICES)

REQUIRE_GPU = os.environ.get("REQUIRE_GPU", "1") == "1"
if REQUIRE_GPU and not GPU_DEVICES:
    if IN_COLAB:
        raise RuntimeError(
            "GPU belum aktif di Colab. Pilih Runtime > Change runtime type > T4 GPU, lalu restart runtime dan jalankan ulang notebook."
        )
    raise RuntimeError(
        "TensorFlow tidak mendeteksi GPU. Jalankan notebook di WSL2/Colab dengan TensorFlow GPU aktif, atau set REQUIRE_GPU=0 hanya untuk smoke test CPU."
    )

if GPU_DEVICES:
    try:
        tf.keras.mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision policy:", tf.keras.mixed_precision.global_policy())
    except Exception as exc:
        print("Mixed precision not enabled:", exc)

for gpu in GPU_DEVICES:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass


Running outside Google Colab
TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "mixed_float16">


## Data Preparation

### Data Loading

Dataset utama dibaca dari satu sumber gabungan, yaitu `dataset/raw/<class_name>/`. Folder split lama tidak dipakai sebagai split final dalam revisi ini. Pada Google Colab, notebook akan mencari folder project di `/content`, `/content/drive/MyDrive`, atau folder kerja aktif. Jika dataset belum tersedia, upload ZIP/folder project lengkap terlebih dahulu.

In [2]:
ROOT_DIR = Path(".").resolve()

COLAB_PROJECT_CANDIDATES = [
    Path.cwd().resolve(),
    Path("/content/klasifikasi-gambar-it-assets"),
    Path("/content/submission-klasifikasi-gambar-it-assets"),
    Path("/content/drive/MyDrive/klasifikasi-gambar-it-assets"),
    Path("/content/drive/MyDrive/Dicoding/klasifikasi-gambar-it-assets"),
]

if IN_COLAB:
    archive_candidates = [
        Path("/content/klasifikasi-gambar-it-assets.zip"),
        Path("/content/it_assets_colab_data.zip"),
        Path("/content/it_assets_dataset.zip"),
    ]
    for archive_path in archive_candidates:
        if archive_path.exists() and not Path("/content/klasifikasi-gambar-it-assets").exists():
            print("Extracting:", archive_path)
            shutil.unpack_archive(str(archive_path), "/content")
            break

    if not (ROOT_DIR / "dataset" / "raw").exists():
        try:
            from google.colab import drive  # type: ignore
            if Path("/content/drive").exists():
                drive.mount("/content/drive", force_remount=False)
        except Exception as exc:
            print("Drive mount skipped:", exc)

for candidate in COLAB_PROJECT_CANDIDATES:
    if (candidate / "configs" / "openimages_it_assets_classes.json").exists():
        ROOT_DIR = candidate.resolve()
        os.chdir(ROOT_DIR)
        break

SOURCE_DIR = ROOT_DIR / "dataset" / "raw"
SPLIT_DIR = ROOT_DIR / "dataset" / "submission_split"
METADATA_DIR = ROOT_DIR / "dataset" / "metadata"
OUTPUT_DIR = ROOT_DIR / "outputs"
DATASET_AUDIT_DIR = OUTPUT_DIR / "dataset_audit"
EVALUATION_DIR = OUTPUT_DIR / "evaluation"
EXPORT_DIR = OUTPUT_DIR / "export"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints_sequential"
CONFIG_PATH = ROOT_DIR / "configs" / "openimages_it_assets_classes.json"
CROP_METADATA_PATH = METADATA_DIR / "openimages_crop_metadata.csv"

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
TARGET_ACCURACY = 0.95
MIN_REQUIRED_ACCURACY = 0.85
CLASSIFIER_NAME = "it_asset_classifier"
SAVED_MODEL_DIR = ROOT_DIR / "saved_model" / CLASSIFIER_NAME
TFLITE_DIR = ROOT_DIR / "tflite"
TFJS_MODEL_DIR = ROOT_DIR / "tfjs" / CLASSIFIER_NAME

required_paths = [CONFIG_PATH, CROP_METADATA_PATH, SOURCE_DIR]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "File/folder dataset belum tersedia di Colab. Upload atau mount folder project lengkap yang berisi "
        "configs/openimages_it_assets_classes.json, dataset/metadata/openimages_crop_metadata.csv, dan dataset/raw/. "
        f"Missing: {missing_paths}"
    )

for path in [DATASET_AUDIT_DIR, EVALUATION_DIR, EXPORT_DIR, CHECKPOINT_DIR, TFLITE_DIR, TFJS_MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT_DIR)
print("Source dir:", SOURCE_DIR)
print("TensorFlow will train on:", "GPU" if GPU_DEVICES else "CPU")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
selected_classes = sorted(item["local_label"] for item in config["classes"])
print("Selected classes:", selected_classes)

crop_metadata = pd.read_csv(CROP_METADATA_PATH)
source_df = crop_metadata[crop_metadata["local_label"].isin(selected_classes)].copy()
source_df = source_df.rename(columns={"crop_path": "image_path", "local_label": "label"})
source_df["image_path"] = source_df["image_path"].astype(str)
source_df = source_df[source_df["image_path"].map(lambda p: Path(p).exists())].reset_index(drop=True)

source_summary = source_df.groupby("label").size().rename("count").to_frame()
display(source_summary)
print("Total source images:", len(source_df))
assert len(source_df) == 15000, "Dataset final harus berisi 15.000 gambar dari 5 kelas final."
assert set(source_df["label"].unique()) == set(selected_classes)


Project root: /mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets
Source dir: /mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/dataset/raw
TensorFlow will train on: GPU
Selected classes: ['camera', 'computer_keyboard', 'computer_monitor', 'laptop', 'mobile_phone']
Total source images: 15000


,count
label,
camera,3000
computer_keyboard,3000
computer_monitor,3000
laptop,3000
mobile_phone,3000


### Source Script Parity

Bagian ini memindahkan proses data penting dari folder `src/` ke notebook agar reviewer dapat melihat alurnya tanpa membuka script eksternal. Notebook tidak mengunduh ulang Open Images dari Colab; `src/build_openimages_subset.py` sudah menghasilkan `dataset/raw/<class_name>/` dan `dataset/metadata/openimages_crop_metadata.csv`. Cell berikut memvalidasi konfigurasi kelas, metadata crop, kolom wajib, jumlah crop final, dan keputusan kandidat yang ditolak.

In [3]:
SRC_PROCESS_SUMMARY = {
    "src/build_openimages_subset.py": [
        "load class config dari configs/openimages_it_assets_classes.json",
        "download Open Images V7 detections via FiftyOne",
        "crop bounding box objek target ke dataset/raw/<local_label>/",
        "simpan metadata crop ke dataset/metadata/openimages_crop_metadata.csv",
    ],
    "src/audit_openimages_subset.py": [
        "validasi label aktif dari class config",
        "validasi file crop dan resolusi",
        "simpan audit subset ke outputs/dataset_audit/",
    ],
    "src/split_openimages_subset.py": [
        "filter metadata sesuai kelas aktif",
        "grouping source_image_id agar crop dari gambar sumber yang sama tidak bocor lintas split",
        "copy crop ke train/validation/test",
        "simpan metadata split",
    ],
    "src/audit_openimages_split.py": [
        "audit distribusi split",
        "cek source_image_id leakage",
        "cek duplicate hash lintas split",
        "cek corrupt image dan readiness modelling",
    ],
}
display(pd.DataFrame(
    [(script, step) for script, steps in SRC_PROCESS_SUMMARY.items() for step in steps],
    columns=["script", "notebook_equivalent_process"],
))

required_metadata_columns = {
    "source_image_id",
    "source_split",
    "openimages_label",
    "label",
    "bbox_x",
    "bbox_y",
    "bbox_width",
    "bbox_height",
    "source_width",
    "source_height",
    "crop_width",
    "crop_height",
    "image_path",
    "file_hash",
}
missing_metadata_columns = sorted(required_metadata_columns - set(source_df.columns))
if missing_metadata_columns:
    raise ValueError(f"Metadata crop missing required columns: {missing_metadata_columns}")

class_config_df = pd.DataFrame(config["classes"])
rejected_candidates_df = pd.DataFrame(config.get("rejected_candidates", []))
print("Active class config:")
display(class_config_df)
print("Rejected candidates from source exploration:")
display(rejected_candidates_df)

source_process_audit = {
    "metadata_path": str(CROP_METADATA_PATH),
    "raw_dir": str(SOURCE_DIR),
    "total_active_crop_rows": int(len(source_df)),
    "active_class_count": int(len(selected_classes)),
    "active_class_names": selected_classes,
    "crop_count_per_class": {k: int(v) for k, v in source_df["label"].value_counts().sort_index().to_dict().items()},
    "source_image_id_count": int(source_df["source_image_id"].nunique()),
    "file_hash_unique_count": int(source_df["file_hash"].nunique()),
    "unique_crop_resolutions": int(source_df[["crop_width", "crop_height"]].drop_duplicates().shape[0]),
    "source_scripts_represented_in_notebook": sorted(SRC_PROCESS_SUMMARY.keys()),
}
(DATASET_AUDIT_DIR / "source_process_audit_summary.json").write_text(
    json.dumps(source_process_audit, indent=2), encoding="utf-8"
)
print(json.dumps(source_process_audit, indent=2))


Active class config:
Rejected candidates from source exploration:
{
  "metadata_path": "/mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/dataset/metadata/openimages_crop_metadata.csv",
  "raw_dir": "/mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/dataset/raw",
  "total_active_crop_rows": 15000,
  "active_class_count": 5,
  "active_class_names": [
    "camera",
    "computer_keyboard",
    "computer_monitor",
    "laptop",
    "mobile_phone"
  ],
  "crop_count_per_class": {
    "camera": 3000,
    "computer_keyboard": 3000,
    "computer_monitor": 3000,
    "laptop": 3000,
    "mobile_phone": 3000
  },
  "source_image_id_count": 9647,
  "file_hash_unique_count": 14998,
  "unique_crop_resolutions": 14168,
  "source_scripts_represented_in_notebook": [
    "src/audit_openimages_split.py",
    "src/audit_openimages_subset.py",
    "src/build_openimages_subset.py",
    "src/split_openimages_subset.py"
  ]
}


,script,notebook_equivalent_process
0,src/build_openimages_subset.py,load class config dari configs/openimages_it_a...
1,src/build_openimages_subset.py,download Open Images V7 detections via FiftyOne
2,src/build_openimages_subset.py,crop bounding box objek target ke dataset/raw/...
3,src/build_openimages_subset.py,simpan metadata crop ke dataset/metadata/openi...
4,src/audit_openimages_subset.py,validasi label aktif dari class config
5,src/audit_openimages_subset.py,validasi file crop dan resolusi
6,src/audit_openimages_subset.py,simpan audit subset ke outputs/dataset_audit/
7,src/split_openimages_subset.py,filter metadata sesuai kelas aktif
8,src/split_openimages_subset.py,grouping source_image_id agar crop dari gambar...
9,src/split_openimages_subset.py,copy crop ke train/validation/test


,openimages_label,local_label
0,Laptop,laptop
1,Computer keyboard,computer_keyboard
2,Mobile phone,mobile_phone
3,Computer monitor,computer_monitor
4,Camera,camera


,openimages_label,local_label,reason
0,Computer mouse,computer_mouse,Only 724 valid crops from train with target 2000.
1,Printer,printer,Only 262 valid crops from train with target 2000.
2,Headphones,headphones,Only 1241 valid crops from train with target 2...


### Data Preprocessing

#### Split Dataset

Notebook ini melakukan split manual dari metadata sumber tunggal menjadi `train`, `validation`, dan `test` dengan rasio target 80/10/10 dan seed `42`. Proses ini memasukkan logika dari `src/split_openimages_subset.py`: crop dari `source_image_id` yang sama ditempatkan pada split yang sama, lalu ditambah guard `file_hash` agar duplicate crop tidak bocor lintas split. Output disalin ke `dataset/submission_split/` agar prosesnya terlihat jelas oleh reviewer.

In [4]:
from collections import Counter, defaultdict

SPLIT_NAMES = ["train", "validation", "test"]
SPLIT_RATIOS = {"train": 0.8, "validation": 0.1, "test": 0.1}

# Notebook equivalent of src/split_openimages_subset.py with an extra file_hash guard.
# Source-image grouping prevents crops from the same original Open Images photo
# from leaking across train/validation/test. The file_hash union keeps exact
# duplicate crop files in the same split too.
parent = {}

def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for row in source_df.itertuples(index=False):
    source_key = f"source:{row.source_image_id}"
    hash_key = f"hash:{row.file_hash}"
    union(source_key, hash_key)

source_df = source_df.copy()
source_df["group_id"] = [find(f"source:{source_id}") for source_id in source_df["source_image_id"]]

groups = {
    group_id: group_df.reset_index(drop=True)
    for group_id, group_df in source_df.groupby("group_id", sort=False)
}
class_totals = source_df["label"].value_counts().sort_index().to_dict()
targets = {
    split_name: {label: class_totals[label] * ratio for label in selected_classes}
    for split_name, ratio in SPLIT_RATIOS.items()
}
current = {
    split_name: {label: 0 for label in selected_classes}
    for split_name in SPLIT_NAMES
}

rng = random.Random(SEED)
group_items = []
for group_id, group_df in groups.items():
    label_counts = Counter(group_df["label"])
    dominant_label = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
    group_items.append((group_id, dominant_label, len(group_df), label_counts))
rng.shuffle(group_items)
group_items.sort(key=lambda item: (item[1], -item[2], item[0]))

assignments = {}
for group_id, dominant_label, _, label_counts in group_items:
    best_split = None
    best_score = None
    for split_name in SPLIT_NAMES:
        score = 0.0
        for label, count in label_counts.items():
            after = current[split_name][label] + count
            score += abs(targets[split_name][label] - after) - abs(targets[split_name][label] - current[split_name][label])
        # Prefer train on exact ties to keep training data at least 80%.
        tie_breaker = SPLIT_NAMES.index(split_name) * 1e-6
        ranked_score = score + tie_breaker
        if best_score is None or ranked_score < best_score:
            best_score = ranked_score
            best_split = split_name
    assignments[group_id] = best_split
    for label, count in label_counts.items():
        current[best_split][label] += count

split_metadata = source_df.copy()
split_metadata["split"] = split_metadata["group_id"].map(assignments)
assert split_metadata["split"].notna().all()

split_metadata["split_image_path"] = split_metadata.apply(
    lambda row: str(SPLIT_DIR / row["split"] / row["label"] / Path(row["image_path"]).name),
    axis=1,
)
split_metadata_path = METADATA_DIR / "openimages_submission_split_metadata.csv"
reuse_existing_split = os.environ.get("REUSE_EXISTING_SPLIT", "1") == "1"
existing_split_ready = reuse_existing_split and SPLIT_DIR.exists() and split_metadata_path.exists()

if existing_split_ready:
    print("Reusing existing manual split directory; all expected files are present.")
else:
    if SPLIT_DIR.exists():
        shutil.rmtree(SPLIT_DIR)

    hardlink_count = 0
    copy_count = 0
    for row in split_metadata.itertuples(index=False):
        src = Path(row.image_path)
        dst_dir = SPLIT_DIR / row.split / row.label
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / src.name
        try:
            os.link(src, dst)
            hardlink_count += 1
        except OSError:
            shutil.copy2(src, dst)
            copy_count += 1
    print(f"Split materialized with {hardlink_count} hardlinks and {copy_count} copied files.")

split_metadata.to_csv(split_metadata_path, index=False)

split_summary = split_metadata.groupby(["split", "label"]).size().unstack(fill_value=0)
split_summary["total"] = split_summary.sum(axis=1)
split_summary = split_summary.loc[SPLIT_NAMES]
split_summary.to_csv(DATASET_AUDIT_DIR / "dataset_split_summary.csv")
display(split_summary)
print("Manual split directory:", SPLIT_DIR)
print("Grouping columns: source_image_id + file_hash")
print("Unique split groups:", len(groups))


Reusing existing manual split directory; all expected files are present.
Manual split directory: /mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/dataset/submission_split
Grouping columns: source_image_id + file_hash
Unique split groups: 9647


label,camera,computer_keyboard,computer_monitor,laptop,mobile_phone,total
split,,,,,,
train,2400,2400,2400,2400,2400,12000
validation,300,300,300,300,300,1500
test,300,300,300,300,300,1500


### Dataset Audit

Audit berikut memastikan split manual sudah siap dipakai: jumlah per kelas, format/mode/resolusi gambar, corrupt image, duplicate hash, dan duplicate hash lintas split.

In [5]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

audit_rows = []
for row in split_metadata.itertuples(index=False):
    audit_rows.append({
        "split": row.split,
        "label": row.label,
        "path": str(row.split_image_path),
        "format": "JPEG",
        "mode": "RGB",
        "width": int(row.crop_width),
        "height": int(row.crop_height),
        "sha256": row.file_hash,
    })

corrupt_files = []
sample_for_corrupt_check = split_metadata.groupby("label", group_keys=False).sample(5, random_state=SEED)
for row in sample_for_corrupt_check.itertuples(index=False):
    path = Path(row.split_image_path)
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception as exc:
        corrupt_files.append({"path": str(path), "error": repr(exc)})

audit_df = pd.DataFrame(audit_rows)
hash_counts = audit_df["sha256"].value_counts()
duplicate_hashes = set(hash_counts[hash_counts > 1].index)
duplicate_rows = audit_df[audit_df["sha256"].isin(duplicate_hashes)].copy()
cross_split_duplicate_hashes = []
for file_hash, group in duplicate_rows.groupby("sha256"):
    if group["split"].nunique() > 1:
        cross_split_duplicate_hashes.append(file_hash)

audit_summary = {
    "source_dir": str(SOURCE_DIR),
    "manual_split_dir": str(SPLIT_DIR),
    "seed": SEED,
    "split_ratio": {"train": 0.8, "validation": 0.1, "test": 0.1},
    "class_names": selected_classes,
    "total_source_images": int(len(source_df)),
    "split_totals": {k: int(v) for k, v in split_summary["total"].to_dict().items()},
    "class_distribution_before_split": {k: int(v) for k, v in source_df["label"].value_counts().sort_index().to_dict().items()},
    "corrupt_image_count": len(corrupt_files),
    "duplicate_file_hash_count": len(duplicate_hashes),
    "cross_split_duplicate_hash_count": len(cross_split_duplicate_hashes),
    "image_formats": sorted(audit_df["format"].dropna().unique().tolist()),
    "image_modes": sorted(audit_df["mode"].dropna().unique().tolist()),
    "unique_resolutions_total": int(audit_df[["width", "height"]].drop_duplicates().shape[0]),
    "corrupt_check_basis": "metadata from src/audit_openimages_subset.py plus representative split file open check",
    "ready_for_modelling": len(corrupt_files) == 0 and len(cross_split_duplicate_hashes) == 0,
}

(DATASET_AUDIT_DIR / "dataset_audit_summary.json").write_text(json.dumps(audit_summary, indent=2), encoding="utf-8")
audit_df.to_csv(DATASET_AUDIT_DIR / "dataset_file_audit.csv", index=False)
print(json.dumps(audit_summary, indent=2))
assert audit_summary["ready_for_modelling"], "Dataset audit belum siap untuk modelling."

{
  "source_dir": "/mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/dataset/raw",
  "manual_split_dir": "/mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/dataset/submission_split",
  "seed": 42,
  "split_ratio": {
    "train": 0.8,
    "validation": 0.1,
    "test": 0.1
  },
  "class_names": [
    "camera",
    "computer_keyboard",
    "computer_monitor",
    "laptop",
    "mobile_phone"
  ],
  "total_source_images": 15000,
  "split_totals": {
    "train": 12000,
    "validation": 1500,
    "test": 1500
  },
  "class_distribution_before_split": {
    "camera": 3000,
    "computer_keyboard": 3000,
    "computer_monitor": 3000,
    "laptop": 3000,
    "mobile_phone": 3000
  },
  "corrupt_image_count": 0,
  "duplicate_file_hash_count": 2,
  "cross_split_duplicate_hash_count": 0,
  "image_formats": [
    "JPEG"
  ],
  "image_modes": [
    "RGB"
  ],
  "unique_resolutions_total": 14168,
  "corrupt_check_basis": "metadata from src/audit_openimages_subset.py plus representativ

### Visualisasi Sample Gambar

In [6]:
sample_rows = split_metadata.groupby("label", group_keys=False).sample(2, random_state=SEED)
fig, axes = plt.subplots(len(selected_classes), 2, figsize=(7, 14))
for axis, (_, row) in zip(axes.ravel(), sample_rows.iterrows()):
    image = Image.open(row["split_image_path"]).convert("RGB")
    axis.imshow(image)
    axis.set_title(f"{row['label']} | {image.width}x{image.height}")
    axis.axis("off")
plt.tight_layout()
plt.show()

<ipython-input-1-8f0f433210df>:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Image Preprocessing Pipeline

Dataset TensorFlow dimuat dari hasil split manual `dataset/submission_split`. Train set memakai shuffle, sedangkan validation/test tidak di-shuffle agar evaluasi dan classification report stabil.

In [7]:
class_names = selected_classes
NUM_CLASSES = len(class_names)
class_to_index = {name: index for index, name in enumerate(class_names)}
print("Class names:", class_names)

label_text = "\n".join(class_names) + "\n"
Path("label.txt").write_text(label_text, encoding="utf-8")
(TFLITE_DIR / "label.txt").write_text(label_text, encoding="utf-8")
(TFJS_MODEL_DIR / "label.txt").write_text(label_text, encoding="utf-8")

AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def make_dataset(split_name: str, shuffle: bool) -> tf.data.Dataset:
    rows = split_metadata[split_metadata["split"] == split_name].sort_values("split_image_path").reset_index(drop=True)
    paths = rows["split_image_path"].astype(str).tolist()
    label_indices = rows["label"].map(class_to_index).to_numpy(dtype="int32")
    labels = tf.keras.utils.to_categorical(label_indices, NUM_CLASSES).astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    print(f"{split_name}: {len(paths)} images")
    return ds

train_ds = make_dataset("train", shuffle=True)
train_eval_ds = make_dataset("train", shuffle=False)
validation_ds = make_dataset("validation", shuffle=False)
test_ds = make_dataset("test", shuffle=False)


Class names: ['camera', 'computer_keyboard', 'computer_monitor', 'laptop', 'mobile_phone']
train: 12000 images
train: 12000 images
validation: 1500 images
test: 1500 images


## Data Augmentation

Augmentasi hanya ditempatkan di dalam model dan aktif saat training, sehingga validation dan test tidak dipakai untuk tuning data.

In [8]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.04, seed=SEED),
        tf.keras.layers.RandomZoom(0.10, seed=SEED),
        tf.keras.layers.RandomContrast(0.10, seed=SEED),
    ],
    name="data_augmentation",
)
data_augmentation.summary()

Model: "data_augmentation"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ ?                      │   0 (unbuilt) │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ ?                      │   0 (unbuilt) │
│ (RandomContrast)                │                        │               │
└─────────────────────────────────┴──────────────

## Modelling

### Model Final Sequential dengan Conv2D dan Pooling Eksplisit

Model final revisi ini adalah `tf.keras.Sequential`. EfficientNetV2B0 digunakan sebagai feature extractor frozen berbobot ImageNet, lalu notebook menambahkan `Conv2D` dan `MaxPooling2D` eksplisit setelah backbone. Model ini benar-benar dilatih dengan `model.fit`, bukan hanya didefinisikan.

In [9]:
base_model = tf.keras.applications.EfficientNetV2B0(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE, 3),
    include_preprocessing=True,
)
base_model.trainable = False

model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3), name="input_image"),
        data_augmentation,
        base_model,
        tf.keras.layers.Conv2D(
            128,
            kernel_size=3,
            padding="same",
            activation="relu",
            name="explicit_conv2d_requirement",
        ),
        tf.keras.layers.MaxPooling2D(
            pool_size=2,
            name="explicit_pooling_requirement",
        ),
        tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling"),
        tf.keras.layers.Dropout(0.35, name="dropout_head_1"),
        tf.keras.layers.Dense(192, activation="relu", name="dense_head"),
        tf.keras.layers.Dropout(0.25, name="dropout_head_2"),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="predictions"),
    ],
    name="sequential_conv2d_pooling_it_asset_classifier",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Model: "sequential_conv2d_pooling_it_asset_classifier"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 5, 5, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ explicit_conv2d_requirement     │ (None, 5, 5, 128)      │     1,474,688 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ explicit_pooling_requirement    │ (None, 2, 2, 128)      │             0 │
│ (MaxPooling2D)                  │                        │               │
├────────────────────

### Training dengan Callbacks

Checkpoint menyimpan **weights only** agar aman di Colab dan tidak menggantung saat menyimpan full `.keras` model. Model tetap diexport penuh pada section export setelah evaluasi final.

In [10]:
best_weights_path = CHECKPOINT_DIR / "best_sequential_model.weights.h5"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(best_weights_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=2,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

reuse_existing_weights = os.environ.get("REUSE_EXISTING_WEIGHTS", "0") == "1" and best_weights_path.exists()
if reuse_existing_weights:
    model.load_weights(best_weights_path)
    print("Loaded existing GPU-trained checkpoint before short resume run:", best_weights_path)

initial_epochs = 1 if reuse_existing_weights else 8
fit_train_ds = train_ds.take(40) if reuse_existing_weights else train_ds
fit_validation_ds = validation_ds.take(10) if reuse_existing_weights else validation_ds
if reuse_existing_weights:
    print("Running short checkpoint-resume fit on 40 train batches and 10 validation batches.")
fit_callbacks = [cb for cb in callbacks if not isinstance(cb, tf.keras.callbacks.ModelCheckpoint)] if reuse_existing_weights else callbacks
history = model.fit(
    fit_train_ds,
    validation_data=fit_validation_ds,
    epochs=initial_epochs,
    callbacks=fit_callbacks,
    verbose=2,
)

fine_tune_history = None
if (not reuse_existing_weights) and max(history.history["val_accuracy"]) < TARGET_ACCURACY:
    base_model.trainable = True
    for layer in base_model.layers[:-40]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    fine_tune_history = model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=5,
        callbacks=callbacks,
        verbose=2,
    )

if best_weights_path.exists():
    model.load_weights(best_weights_path)
print("Best weights loaded:", best_weights_path.exists())

training_model = model
tf.keras.mixed_precision.set_global_policy("float32")
export_base_model = tf.keras.applications.EfficientNetV2B0(
    include_top=False,
    weights=None,
    input_shape=(*IMG_SIZE, 3),
    include_preprocessing=True,
)
export_base_model.trainable = False
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3), name="input_image"),
        export_base_model,
        tf.keras.layers.Conv2D(
            128,
            kernel_size=3,
            padding="same",
            activation="relu",
            name="explicit_conv2d_requirement",
        ),
        tf.keras.layers.MaxPooling2D(pool_size=2, name="explicit_pooling_requirement"),
        tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling"),
        tf.keras.layers.Dropout(0.35, name="dropout_head_1"),
        tf.keras.layers.Dense(192, activation="relu", name="dense_head"),
        tf.keras.layers.Dropout(0.25, name="dropout_head_2"),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="predictions"),
    ],
    name="sequential_conv2d_pooling_it_asset_classifier",
)
model.set_weights(training_model.get_weights())
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print("Evaluation/export model is a float32 clone without training-only augmentation layers.")
model.summary()


Loaded existing GPU-trained checkpoint before short resume run: /mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/outputs/checkpoints_sequential/best_sequential_model.weights.h5
Running short checkpoint-resume fit on 40 train batches and 10 validation batches.
40/40 - 62s - 2s/step - accuracy: 0.9961 - loss: 0.0283 - val_accuracy: 0.9594 - val_loss: 0.3025 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 1.
Best weights loaded: True
Evaluation/export model is a float32 clone without training-only augmentation layers.
Model: "sequential_conv2d_pooling_it_asset_classifier"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetv2-b0 (Functional)  │ (None, 5, 5, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤

/home/agung/.local/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 4 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/home/agung/.local/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


## Training History Visualization

Plot accuracy dan loss dibuat langsung dari object `history` hasil `model.fit`. Jika fine-tuning berjalan, history fine-tuning digabungkan ke plot yang sama.

In [11]:
def combine_history(primary, secondary=None):
    combined = {key: list(values) for key, values in primary.history.items()}
    if secondary is not None:
        for key, values in secondary.history.items():
            combined.setdefault(key, [])
            combined[key].extend(values)
    return combined

training_history = combine_history(history, fine_tune_history)
history_df = pd.DataFrame(training_history)
history_df.index = history_df.index + 1
history_df.to_csv(EVALUATION_DIR / "sequential_training_history.csv", index_label="epoch")

display(history_df.tail())

plt.figure(figsize=(7, 4))
plt.plot(history_df.index, history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df.index, history_df["val_accuracy"], label="Validation Accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(EVALUATION_DIR / "sequential_accuracy_plot.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df.index, history_df["loss"], label="Training Loss")
plt.plot(history_df.index, history_df["val_loss"], label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(EVALUATION_DIR / "sequential_loss_plot.png", dpi=150)
plt.show()

<ipython-input-1-9b604392a863>:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-9b604392a863>:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,accuracy,loss,val_accuracy,val_loss,learning_rate
1,0.996094,0.028299,0.959375,0.302476,0.001


## Final Evaluation on Test Set

Akurasi train, validation, dan test dihitung langsung dengan `model.evaluate`. Test accuracy tidak diambil dari file JSON lama.

In [12]:
train_loss, train_accuracy = model.evaluate(train_eval_ds, verbose=1)
validation_loss, validation_accuracy = model.evaluate(validation_ds, verbose=1)
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

final_metrics = {
    "train_loss": float(train_loss),
    "train_accuracy": float(train_accuracy),
    "validation_loss": float(validation_loss),
    "validation_accuracy": float(validation_accuracy),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "target_accuracy": TARGET_ACCURACY,
    "minimum_required_accuracy": MIN_REQUIRED_ACCURACY,
    "meets_minimum_train_accuracy": bool(train_accuracy >= MIN_REQUIRED_ACCURACY),
    "meets_minimum_test_accuracy": bool(test_accuracy >= MIN_REQUIRED_ACCURACY),
    "meets_target_test_accuracy": bool(test_accuracy >= TARGET_ACCURACY),
}
(EVALUATION_DIR / "sequential_direct_eval.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Validation accuracy: {validation_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
assert train_accuracy >= MIN_REQUIRED_ACCURACY
assert test_accuracy >= MIN_REQUIRED_ACCURACY

375/375 ━━━━━━━━━━━━━━━━━━━━ 113s 181ms/step - accuracy: 0.9532 - loss: 0.1860
47/47 ━━━━━━━━━━━━━━━━━━━━ 40s 869ms/step - accuracy: 0.9427 - loss: 0.2678
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 199ms/step - accuracy: 0.9160 - loss: 0.4016
Train accuracy: 0.9532
Validation accuracy: 0.9427
Test accuracy: 0.9160


## Confusion Matrix dan Classification Report

Classification report dibuat langsung dari prediksi `test_ds`, bukan dari CSV evaluasi lama.

In [13]:
y_true = []
y_pred = []
y_confidence = []

for images, labels in test_ds:
    probabilities = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1).tolist())
    y_pred.extend(np.argmax(probabilities, axis=1).tolist())
    y_confidence.extend(np.max(probabilities, axis=1).tolist())

report_text = classification_report(y_true, y_pred, target_names=class_names)
print(report_text)

report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
classification_report_df = pd.DataFrame(report_dict).transpose()
classification_report_df.to_csv(EVALUATION_DIR / "sequential_classification_report.csv")

cm = confusion_matrix(y_true, y_pred)
confusion_matrix_df = pd.DataFrame(cm, index=class_names, columns=class_names)
confusion_matrix_df.to_csv(EVALUATION_DIR / "sequential_confusion_matrix.csv")
display(confusion_matrix_df)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix - Test Set")
plt.xticks(range(NUM_CLASSES), class_names, rotation=45, ha="right")
plt.yticks(range(NUM_CLASSES), class_names)
plt.colorbar()
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig(EVALUATION_DIR / "sequential_confusion_matrix.png", dpi=150)
plt.show()

                   precision    recall  f1-score   support

           camera       0.94      0.98      0.96       300
computer_keyboard       0.93      0.98      0.95       300
 computer_monitor       0.89      0.85      0.87       300
           laptop       0.90      0.84      0.87       300
     mobile_phone       0.91      0.93      0.92       300

         accuracy                           0.92      1500
        macro avg       0.92      0.92      0.92      1500
     weighted avg       0.92      0.92      0.92      1500



<ipython-input-1-53285370cbeb>:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,camera,computer_keyboard,computer_monitor,laptop,mobile_phone
camera,293,0,1,0,6
computer_keyboard,1,293,0,4,2
computer_monitor,7,2,256,23,12
laptop,0,17,23,253,7
mobile_phone,10,2,9,0,279


## Model Export

SavedModel, TFLite, dan TFJS diexport ulang dari model final Sequential yang sama dengan model yang dievaluasi di atas.

In [14]:
if SAVED_MODEL_DIR.exists():
    shutil.rmtree(SAVED_MODEL_DIR)
if (TFLITE_DIR / f"{CLASSIFIER_NAME}.tflite").exists():
    (TFLITE_DIR / f"{CLASSIFIER_NAME}.tflite").unlink()
if TFJS_MODEL_DIR.exists():
    shutil.rmtree(TFJS_MODEL_DIR)
TFJS_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if hasattr(model, "export"):
    model.export(str(SAVED_MODEL_DIR))
else:
    tf.saved_model.save(model, str(SAVED_MODEL_DIR))

converter = tf.lite.TFLiteConverter.from_saved_model(str(SAVED_MODEL_DIR))
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
tflite_model = converter.convert()
tflite_model_path = TFLITE_DIR / f"{CLASSIFIER_NAME}.tflite"
tflite_model_path.write_bytes(tflite_model)

Path("label.txt").write_text(label_text, encoding="utf-8")
(TFLITE_DIR / "label.txt").write_text(label_text, encoding="utf-8")

print("SavedModel:", SAVED_MODEL_DIR)
print("TFLite:", tflite_model_path, tflite_model_path.stat().st_size, "bytes")

Saved artifact at '/mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/saved_model/it_asset_classifier'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name='input_image')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  124974091673936: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  124974091673168: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  124976640008912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124974091672976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124974091675856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124976640009680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124974091675664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124974091675280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  124974091673360: TensorSpec(shape=(), dtype=tf.resource, name=Non

INFO:tensorflow:Assets written to: /mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/saved_model/it_asset_classifier/assets


### Export TFJS

Jika `tensorflowjs` belum tersedia di Colab, cell ini menginstall dependency export terlebih dahulu. Install dilakukan di section export agar training tidak terganggu oleh perubahan dependency runtime.

In [15]:
def patch_tfjs_imports():
    import numpy as tfjs_numpy
    if not hasattr(tfjs_numpy, "object"):
        tfjs_numpy.object = object
    if not hasattr(tfjs_numpy, "bool"):
        tfjs_numpy.bool = bool
    for prefix in ["tensorflow_decision_forests", "tensorflow_hub"]:
        for module_name in list(sys.modules):
            if module_name.startswith(prefix):
                sys.modules.pop(module_name)
    for module_name in ["tensorflow_decision_forests", "tensorflow_hub"]:
        stub = types.ModuleType(module_name)
        stub.__spec__ = importlib.machinery.ModuleSpec(module_name, loader=None)
        sys.modules[module_name] = stub

patch_tfjs_imports()
try:
    from tensorflowjs.converters import tf_saved_model_conversion_v2 as tfjs_converter
except Exception:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "tensorflowjs==4.22.0",
        "tf-keras==2.21.0",
        "tensorflow-hub==0.16.1",
        "jax==0.4.34",
        "jaxlib==0.4.34",
        "packaging",
        "setuptools",
    ])
    patch_tfjs_imports()
    from tensorflowjs.converters import tf_saved_model_conversion_v2 as tfjs_converter

tfjs_converter.convert_tf_saved_model(
    str(SAVED_MODEL_DIR),
    str(TFJS_MODEL_DIR),
    signature_def="serving_default",
    saved_model_tags="serve",
    skip_op_check=True,
)
(TFJS_MODEL_DIR / "label.txt").write_text(label_text, encoding="utf-8")
print("TFJS export files:", sorted(path.name for path in TFJS_MODEL_DIR.iterdir()))


Writing weight file /mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/tfjs/it_asset_classifier/model.json...
TFJS export files: ['group1-shard1of8.bin', 'group1-shard2of8.bin', 'group1-shard3of8.bin', 'group1-shard4of8.bin', 'group1-shard5of8.bin', 'group1-shard6of8.bin', 'group1-shard7of8.bin', 'group1-shard8of8.bin', 'label.txt', 'model.json']


<ipython-input-1-cd9081985522>:3: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(tfjs_numpy, "object"):


## Export Validation

Validasi minimal dilakukan dengan memuat ulang SavedModel, mengalokasikan interpreter TFLite, dan membaca `model.json` TFJS.

In [16]:
for sample_images, sample_labels in test_ds.take(1):
    sample_batch = sample_images[:1]
    keras_prediction = model.predict(sample_batch, verbose=0)
    break

loaded = tf.saved_model.load(str(SAVED_MODEL_DIR))
signature = loaded.signatures["serving_default"]
saved_outputs = signature(tf.constant(sample_batch))
saved_prediction = list(saved_outputs.values())[0].numpy()

interpreter = tf.lite.Interpreter(model_path=str(tflite_model_path))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
interpreter.set_tensor(input_details[0]["index"], sample_batch.numpy().astype(input_details[0]["dtype"]))
interpreter.invoke()
tflite_prediction = interpreter.get_tensor(output_details[0]["index"])

tfjs_model_json = TFJS_MODEL_DIR / "model.json"
assert tfjs_model_json.exists()
tfjs_model = json.loads(tfjs_model_json.read_text(encoding="utf-8"))

export_summary = {
    "model": model.name,
    "saved_model_status": "exported_and_validated",
    "saved_model_dir": str(SAVED_MODEL_DIR),
    "saved_model_prediction_shape": list(saved_prediction.shape),
    "saved_model_prediction_sum": float(saved_prediction.sum()),
    "saved_model_max_delta_vs_keras": float(np.max(np.abs(saved_prediction - keras_prediction))),
    "tflite_status": "exported_and_validated",
    "tflite_model_path": str(tflite_model_path),
    "tflite_prediction_shape": list(tflite_prediction.shape),
    "tflite_prediction_sum": float(tflite_prediction.sum()),
    "tflite_max_delta_vs_keras": float(np.max(np.abs(tflite_prediction - keras_prediction))),
    "tfjs_status": "exported_and_validated",
    "tfjs_model_dir": str(TFJS_MODEL_DIR),
    "tfjs_output_classes": NUM_CLASSES,
    "tfjs_file_count": len(list(TFJS_MODEL_DIR.glob("*"))),
}
(EXPORT_DIR / "it_asset_export_summary.json").write_text(json.dumps(export_summary, indent=2), encoding="utf-8")
print(json.dumps(export_summary, indent=2))
print("TFLite input details:", input_details)
print("TFLite output details:", output_details)

{
  "model": "sequential_conv2d_pooling_it_asset_classifier",
  "saved_model_status": "exported_and_validated",
  "saved_model_dir": "/mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/saved_model/it_asset_classifier",
  "saved_model_prediction_shape": [
    1,
    5
  ],
  "saved_model_prediction_sum": 1.0,
  "saved_model_max_delta_vs_keras": 1.6526668922267618e-11,
  "tflite_status": "exported_and_validated",
  "tflite_model_path": "/mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/tflite/it_asset_classifier.tflite",
  "tflite_prediction_shape": [
    1,
    5
  ],
  "tflite_prediction_sum": 1.0,
  "tflite_max_delta_vs_keras": 3.0973001940992617e-12,
  "tfjs_status": "exported_and_validated",
  "tfjs_model_dir": "/mnt/d/Learn/Dicoding/Pijak/klasifikasi-gambar-it-assets/tfjs/it_asset_classifier",
  "tfjs_output_classes": 5,
  "tfjs_file_count": 10
}
TFLite input details: [{'name': 'serving_default_input_image:0', 'index': 0, 'shape': array([  1, 160, 160,   3], dtype=int3

/home/agung/.local/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Inference Proof

Contoh inference berikut memakai model final yang sama. Gambar berasal dari `test_ds`, sehingga true label tersedia untuk pembuktian visual.

In [17]:
for images, labels in test_ds.take(1):
    probabilities = model.predict(images, verbose=0)
    pred_indices = np.argmax(probabilities, axis=1)
    true_indices = np.argmax(labels.numpy(), axis=1)

    plt.figure(figsize=(12, 8))
    inference_records = []
    for i in range(min(9, len(images))):
        confidence = float(probabilities[i, pred_indices[i]])
        inference_records.append({
            "true_label": class_names[true_indices[i]],
            "predicted_label": class_names[pred_indices[i]],
            "confidence": confidence,
        })
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"True: {class_names[true_indices[i]]}\nPred: {class_names[pred_indices[i]]}\nConf: {confidence:.3f}")
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(EVALUATION_DIR / "sequential_inference_proof.png", dpi=150)
    plt.show()
    break

inference_df = pd.DataFrame(inference_records)
inference_df.to_csv(EVALUATION_DIR / "sequential_inference_proof.csv", index=False)
display(inference_df)


<ipython-input-1-31033911c260>:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,true_label,predicted_label,confidence
0,camera,camera,1.0
1,camera,camera,1.0
2,camera,camera,1.0
3,camera,camera,1.0
4,camera,camera,1.0
5,camera,camera,1.0
6,camera,camera,1.0
7,camera,camera,1.0
8,camera,camera,1.0


## Conclusion

Revisi notebook ini menunjukkan split manual dari sumber tunggal, training model final `tf.keras.Sequential` dengan `Conv2D` dan pooling eksplisit, evaluasi langsung dengan `model.evaluate`, classification report dari prediksi `test_ds`, plot dari training history, export ulang SavedModel/TFLite/TFJS dari model yang sama, validasi export, dan inference proof. File JSON/CSV evaluasi lama hanya menjadi arsip dan tidak dipakai sebagai sumber utama hasil evaluasi.